# Zero-shot forecasting with Mistral's Timestral vs classical baselines

**Question:** can a 28M-parameter time-series foundation model, used *zero-shot* (no training on this data), compete with a tuned classical model on a strongly seasonal monthly series?

**Data:** US monthly electricity production index (FRED series `IPG2211A2N`, via Kaggle), Jan 1985 – Jan 2018.

**Models compared**
1. **Seasonal naive** – next 12 months = last 12 months
2. **SARIMA(1,1,1)(1,1,1,12)** – fitted on the training window
3. **Timestral-28M** – Mistral's time-series foundation model, zero-shot

**Evaluation**
- Single 12-month hold-out
- 5-fold rolling-origin backtest (expanding window, 12-month horizon) for a more honest comparison
- Timestral's probabilistic output (80% / 90% prediction intervals)

## 1. Setup

In [ ]:
# Run once, then restart the kernel
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from timestral import TimestralForecaster

DATA_PATH = Path("data/Electric_Production.csv")
FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

HORIZON = 12  # months

In [ ]:
forecaster = TimestralForecaster.from_pretrained("mistralai/Timestral-28M")

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"IPG2211A2N": "Electricity_Production"})

print(df.shape)
print(df.dtypes)
df.head(10)

In [ ]:
df.info()
df.describe(include="all")

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isna().sum())

In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"])
df = df.sort_values("DATE").set_index("DATE")
series = df["Electricity_Production"]

print(f"{series.index.min().date()} to {series.index.max().date()} ({len(series)} months)")

## 3. Exploratory analysis

In [ ]:
ax = series.plot(figsize=(14, 5))
ax.set_title("Electricity Production Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Electricity Production (index)")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_series.png", dpi=150)
plt.show()

In [ ]:
decomposition = seasonal_decompose(series, model="additive", period=12)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_decomposition.png", dpi=150)
plt.show()

Strong upward trend up to the late 2000s, then a plateau, with a clear double seasonal peak each year (winter heating and summer cooling).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(series, lags=36, ax=axes[0])
plot_pacf(series, lags=36, ax=axes[1])
plt.tight_layout()
plt.savefig(FIG_DIR / "03_acf_pacf.png", dpi=150)
plt.show()

## 4. Metrics

In [ ]:
def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": np.mean(np.abs((actual - predicted) / actual)) * 100,
    }

## 5. Single 12-month hold-out

The last 12 months are hidden from every model.

In [ ]:
train = series.iloc[:-HORIZON]
test = series.iloc[-HORIZON:]

print("Training:", len(train))
print("Test:", len(test))
test

In [ ]:
# 1. Seasonal naive
naive_forecast = train.iloc[-12:].values

# 2. SARIMA
sarima_fit = SARIMAX(
    train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)
).fit(disp=False)
sarima_forecast = sarima_fit.forecast(steps=HORIZON).values

# 3. Timestral (zero-shot: only sees the raw history)
holdout_forecast = forecaster.forecast(train.values, prediction_length=HORIZON)
timestral_forecast = holdout_forecast.median.detach().cpu().numpy().squeeze()

In [ ]:
holdout_results = pd.DataFrame({
    "Seasonal Naive": metrics(test.values, naive_forecast),
    "SARIMA": metrics(test.values, sarima_forecast),
    "Timestral-28M": metrics(test.values, timestral_forecast),
}).T.round(3)

holdout_results.to_csv(RESULTS_DIR / "holdout_metrics.csv")
holdout_results

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test.values, label="Actual", color="black", marker="o", linewidth=3)
plt.plot(test.index, naive_forecast, label="Seasonal Naive", linestyle="--")
plt.plot(test.index, sarima_forecast, label="SARIMA", marker="o")
plt.plot(test.index, timestral_forecast, label="Timestral-28M", marker="o")
plt.title("12-Month Hold-out Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Electricity Production (index)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_holdout_comparison.png", dpi=150)
plt.show()

## 6. Rolling-origin backtest

One hold-out year can flatter or punish any model by chance. Here every model is re-evaluated on each of the last 5 years, with an expanding training window. SARIMA is refitted per fold; Timestral just receives a longer context.

In [ ]:
def rolling_backtest(series, forecaster, horizon=12, step=12, n_folds=5):
    results, predictions = [], []

    # Work backwards from the end so the last n_folds years are each a test window
    first_test_start = len(series) - (horizon + (n_folds - 1) * step)

    for fold in range(n_folds):
        test_start = first_test_start + fold * step
        train_fold = series.iloc[:test_start]
        test_fold = series.iloc[test_start:test_start + horizon]
        actual = test_fold.values

        print(f"Fold {fold + 1}: {test_fold.index[0].date()} to "
              f"{test_fold.index[-1].date()} | train size {len(train_fold)}")

        naive_pred = train_fold.iloc[-12:].values

        sarima_pred = (
            SARIMAX(train_fold, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
            .fit(disp=False)
            .forecast(steps=horizon)
            .values
        )

        fc = forecaster.forecast(train_fold.values, prediction_length=horizon)
        timestral_pred = fc.median.detach().cpu().numpy().squeeze()

        for name, pred in [
            ("Seasonal Naive", naive_pred),
            ("SARIMA", sarima_pred),
            ("Timestral-28M", timestral_pred),
        ]:
            results.append({
                "Fold": fold + 1,
                "Test_Start": test_fold.index[0],
                "Test_End": test_fold.index[-1],
                "Train_Size": len(train_fold),
                "Model": name,
                **metrics(actual, pred),
            })

        for i, date in enumerate(test_fold.index):
            predictions.append({
                "Fold": fold + 1,
                "Date": date,
                "Actual": actual[i],
                "Seasonal_Naive": naive_pred[i],
                "SARIMA": sarima_pred[i],
                "Timestral": timestral_pred[i],
            })

    return pd.DataFrame(results), pd.DataFrame(predictions)

In [ ]:
results_df, predictions_df = rolling_backtest(
    series=series, forecaster=forecaster, horizon=HORIZON, step=12, n_folds=5
)
results_df.to_csv(RESULTS_DIR / "backtest_by_fold.csv", index=False)
results_df

In [ ]:
summary = (
    results_df
    .groupby("Model")[["MAE", "RMSE", "MAPE"]]
    .agg(["mean", "std"])
    .round(3)
    .sort_values(("MAPE", "mean"))
)
summary.to_csv(RESULTS_DIR / "backtest_summary.csv")
summary

In [ ]:
# Markdown version of the summary, ready to paste into the README
flat = summary.copy()
flat.columns = [f"{m} ({s})" for m, s in flat.columns]
print(flat.to_markdown())

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(predictions_df["Date"], predictions_df["Actual"],
         label="Actual", color="black", linewidth=2.5, marker="o")
plt.plot(predictions_df["Date"], predictions_df["Seasonal_Naive"],
         label="Seasonal Naive", linestyle="--")
plt.plot(predictions_df["Date"], predictions_df["SARIMA"], label="SARIMA")
plt.plot(predictions_df["Date"], predictions_df["Timestral"], label="Timestral-28M")
plt.title("Rolling 12-Month Forecasts (5 folds)")
plt.xlabel("Date")
plt.ylabel("Electricity Production (index)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_rolling_backtest.png", dpi=150)
plt.show()

## 7. Probabilistic forecast

Timestral returns 21 quantiles, not just a point forecast, so we get prediction intervals for free.

In [ ]:
print(holdout_forecast.quantile_levels)
print("Quantiles shape:", holdout_forecast.quantiles.shape)

q = {
    level: holdout_forecast.quantile(level).detach().cpu().numpy().squeeze()
    for level in [0.05, 0.10, 0.50, 0.90, 0.95]
}

# Empirical coverage on the hold-out year
for lo, hi in [(0.05, 0.95), (0.10, 0.90)]:
    inside = ((test.values >= q[lo]) & (test.values <= q[hi])).mean()
    print(f"{int((hi - lo) * 100)}% interval covered {inside:.0%} of actuals")

In [ ]:
history = train.iloc[-36:]

plt.figure(figsize=(14, 6))
plt.plot(history.index, history.values, color="black", linewidth=2, label="Historical")
plt.plot(test.index, test.values, color="black", marker="o", linestyle="--",
         linewidth=2, label="Actual (hidden)")
plt.fill_between(test.index, q[0.05], q[0.95], color="green", alpha=0.10, label="90% interval")
plt.fill_between(test.index, q[0.10], q[0.90], color="green", alpha=0.20, label="80% interval")
plt.plot(test.index, q[0.50], color="green", marker="o", linewidth=2.5, label="Timestral median")
plt.axvline(test.index[0], color="grey", linestyle="--")
plt.title("Timestral-28M Probabilistic Forecast – Electricity Production")
plt.xlabel("Date")
plt.ylabel("Electricity Production (index)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_probabilistic_forecast.png", dpi=150)
plt.show()

## 8. Takeaways

_Fill in once the notebook has run, e.g.:_
- How Timestral-28M ranked against SARIMA and seasonal naive on mean MAPE across folds
- How stable each model was (std across folds)
- Whether the 80% / 90% intervals covered the actuals as expected
- Caveat: one series, one horizon. Foundation models shine most when you have many series and no time to tune each one.